### Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

### Summarization Middlware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


### Messagebased summarization

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
config={"configurable":{"thred_id":"test-1"}}

In [4]:
questions = [
    "What is 2*2",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "what is 3*3",
    "what is 4*4"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},
                            config={"configurable":{"thread_id":"1"}})
    print(f"Messages: {response}")
    print(f"Messages:{len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2*2', additional_kwargs={}, response_metadata={}, id='767b6920-04c7-4878-910a-2f8f10ea3eef'), AIMessage(content='2\u202f×\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "What is 2*2". Answer is 4. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 77, 'total_tokens': 121, 'completion_time': 0.092035068, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.003406788, 'prompt_tokens_details': None, 'queue_time': 0.283238901, 'total_time': 0.095441856}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ff6f4-85b0-74d1-b7a1-f7e2ede687d7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 44, 'total_tokens': 121, 'output_token_details': {'r

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hostels(city: str) -> str:
    '''Search hotels - return long response to use more tokens'''
    return f'''Hotels is {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. city Inn - 4 star, $100/night, bussiness center
    3. Budget stay - 3 star, $75/night, free wifi'''


agent=create_agent(
    model='groq:openai/gpt-oss-120b',
    tools=[search_hostels],
    checkpointer=InMemorySaver(),
    middleware={
        SummarizationMiddleware(
            model='groq:openai/gpt-oss-120b',
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    }
)


config={"configurable":{"thred_id":"test-1"}}


# Token counter

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

In [8]:
# Run test 
cities = ['Paris','London','tokyo','new york','dubai','singapore']

for city in cities :
    config = {"configurable": {"thread_id": "test-1"}}

    # Run test 
    cities = ['Paris','London','tokyo','new york','dubai','singapore']

    for city in cities:
        response = agent.invoke(
            {"messages":[HumanMessage(content=f"find hotels in {city}")]},
            config=config
        )

        tokens = count_tokens(response["messages"])
        print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
        print(f"{(response['messages'])}")

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~1547 tokens, 4 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='3a500195-7949-4d7b-b7e1-fe12d71ce8e8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Paris. We have a function search_hostels (though named hostels, but we can use for hotels). Likely we need to call it with city "Paris". Then present results.', 'tool_calls': [{'id': 'fc_f8d5317d-7706-4fb6-ae44-0b0997cafddf', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hostels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 129, 'total_tokens': 201, 'completion_time': 0.153450487, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.053313802, 'prompt_tokens_details': None, 'queue_time': 0.299035894, 'total_time': 0.206764289}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4140daa9c2', 'service_tier': 'on_demand', 'finish_reason'

KeyboardInterrupt: 